In [36]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

In [37]:
from google.colab import files

uploaded = files.upload()

Saving BT2024033_test_var2.csv to BT2024033_test_var2 (1).csv
Saving BT2024033_train_var2.csv to BT2024033_train_var2 (1).csv


In [38]:
train_df = pd.read_csv("BT2024033_train_var2.csv")
test_df = pd.read_csv("BT2024033_test_var2.csv")

In [39]:
print(train_df.head())
print(train_df.shape)

print(test_df.head())
print(test_df.shape)

print(train_df.columns)
print(test_df.columns)

         x1        x2        x3         y
0 -0.060205 -0.469292  0.897803 -0.751069
1 -1.000000  1.000000  0.146125  5.884484
2  0.810730 -1.000000  0.278668 -2.310624
3 -0.051611 -0.582040  0.413985  0.532068
4  0.907898 -0.225511 -1.000000  1.491205
(1000, 4)
         x1        x2        x3
0  0.843253  0.732663  0.668876
1  1.000000 -0.993574 -0.911081
2 -0.644480  0.352778 -0.473819
3  0.481034 -0.447893 -1.000000
4 -0.358851 -0.554895  0.331569
(1000, 3)
Index(['x1', 'x2', 'x3', 'y'], dtype='object')
Index(['x1', 'x2', 'x3'], dtype='object')


In [40]:
X = train_df[["x1", "x2", "x3"]]
y = train_df["y"]

X_test = test_df[["x1", "x2", "x3"]]

In [41]:
kf = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [42]:
ridge_pipeline = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

In [43]:
ridge_param_grid = {
    "poly__degree": range(1, 21),
    "ridge__alpha": np.logspace(-4, 2, 20)
}

In [44]:
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

In [45]:
ridge_search.fit(X, y)

Fitting 10 folds for each of 400 candidates, totalling 4000 fits


GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('scaler', StandardScaler()),
                                       ('ridge', Ridge())]),
             n_jobs=-1,
             param_grid={'poly__degree': range(1, 21),
                         'ridge__alpha': array([1.00000000e-04, 2.06913808e-04, 4.28133240e-04, 8.85866790e-04,
       1.83298071e-03, 3.79269019e-03, 7.84759970e-03, 1.62377674e-02,
       3.35981829e-02, 6.95192796e-02, 1.43844989e-01, 2.97635144e-01,
       6.15848211e-01, 1.27427499e+00, 2.63665090e+00, 5.45559478e+00,
       1.12883789e+01, 2.33572147e+01, 4.83293024e+01, 1.00000000e+02])},
             scoring='neg_mean_squared_error', verbose=1)

In [46]:
print("Best Ridge parameters:")
print(ridge_search.best_params_)

Best Ridge parameters:
{'poly__degree': 12, 'ridge__alpha': np.float64(1.2742749857031321)}


In [47]:
ridge_cv_mse = -ridge_search.best_score_

print("Best Ridge CV MSE:", ridge_cv_mse)

Best Ridge CV MSE: 0.23850094861416787


In [48]:
lasso_pipeline = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=10000))
])

In [49]:
lasso_param_grid = {
    "poly__degree": range(4, 14),
    "lasso__alpha": np.logspace(-4, -2,5)
}

In [50]:
lasso_search = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_param_grid,
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

lasso_search.fit(X, y)

Fitting 10 folds for each of 50 candidates, totalling 500 fits


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.924e+00, tolerance: 4.842e+00
  model = cd_fast.enet_coordinate_descent(


GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('scaler', StandardScaler()),
                                       ('lasso', Lasso(max_iter=10000))]),
             n_jobs=-1,
             param_grid={'lasso__alpha': array([0.0001    , 0.00031623, 0.001     , 0.00316228, 0.01      ]),
                         'poly__degree': range(4, 14)},
             scoring='neg_mean_squared_error', verbose=1)

In [51]:
print("Best Lasso parameters:")
print(lasso_search.best_params_)

Best Lasso parameters:
{'lasso__alpha': np.float64(0.00031622776601683794), 'poly__degree': 12}


In [52]:
lasso_cv_mse = -lasso_search.best_score_

print("Best Lasso CV MSE:", lasso_cv_mse)

Best Lasso CV MSE: 0.24016242949461994


In [53]:
print("Ridge CV MSE :", ridge_cv_mse)
print("Lasso CV MSE :", lasso_cv_mse)

Ridge CV MSE : 0.23850094861416787
Lasso CV MSE : 0.24016242949461994


In [54]:
from sklearn.linear_model import ElasticNet

In [55]:
elastic_pipeline = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("elastic", ElasticNet(
        max_iter=3000,
        tol=1e-3,
        selection="random"
    ))
])

elastic_param_grid = {
    "poly__degree": [10, 11, 12, 13, 14],
    "elastic__alpha": [0.01, 0.03, 0.1, 0.3],
    "elastic__l1_ratio": [0.2, 0.5, 0.8]
}

elastic_search = GridSearchCV(
    estimator=elastic_pipeline,
    param_grid=elastic_param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

elastic_search.fit(X, y)

elastic_cv_mse = -elastic_search.best_score_

print("Best parameters:", elastic_search.best_params_)
print("Elastic Net CV MSE:", elastic_cv_mse)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best parameters: {'elastic__alpha': 0.01, 'elastic__l1_ratio': 0.2, 'poly__degree': 12}
Elastic Net CV MSE: 0.303274315258158


In [56]:
print("Ridge Training R²:", ridge_search.best_estimator_.score(X, y))

print("Lasso Training R²:", lasso_search.best_estimator_.score(X, y))

print("Elastic Net Training R²:", elastic_search.best_estimator_.score(X, y))

Ridge Training R²: 0.996680318645478
Lasso Training R²: 0.9967149369396008
Elastic Net Training R²: 0.995473580777161


In [57]:
#Now We know thw best Model is Ridge acc to least CV MSE

In [58]:
best_model = ridge_search.best_estimator_

In [59]:
X_test = test_df[["x1", "x2", "x3"]]

In [60]:
print(X.columns)
print(X_test.columns)

Index(['x1', 'x2', 'x3'], dtype='object')
Index(['x1', 'x2', 'x3'], dtype='object')


In [61]:
test_pred = best_model.predict(X_test)

In [62]:
print(test_pred[:10])

[  7.2252852  -21.2971543   -1.2768911   -6.35577998   0.91571548
   1.31442621 -10.9390065    6.78029783   3.78474526  -3.93035705]


In [63]:
output = pd.DataFrame({
    "y": test_pred
})

In [64]:
print(output.head())
print(output.shape)

           y
0   7.225285
1 -21.297154
2  -1.276891
3  -6.355780
4   0.915715
(1000, 1)


In [65]:
output.to_csv("predictions.csv", index=False)

In [66]:
check = pd.read_csv("predictions.csv")

print(check.head())
print(check.shape)

           y
0   7.225285
1 -21.297154
2  -1.276891
3  -6.355780
4   0.915715
(1000, 1)


In [67]:
files.download("predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>